
# Bank Marketing Campaign Prediction Analysis

This notebook analyzes direct marketing campaign data from a Portuguese banking institution to predict term deposit subscriptions (`y`).

The analysis covers:
- **Data exploration** – understanding the dataset, class balance, and key features
- **Baseline model** – logistic regression with full evaluation metrics
- **Hyperparameter optimisation** – grid search to improve performance
- **Regularisation study** – L1 (Lasso) and L2 (Ridge) variants
- **Model comparison** – head-to-head evaluation across all logistic models
- **Decision threshold tuning** – choosing the right operating point for the business objective
- **K-Nearest Neighbours** – alternative classifier, tuned and compared against logistic models
- **Client segmentation** – feature importance analysis and customer profiling for actionable insights


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

RANDOM_STATE = 42
AGE_BINS = [16, 30, 40, 50, 60, 100]
sns.set_theme(style='whitegrid')


## Load dataset and exploratory analysis

In [ ]:
DATA_PATH = 'bank-additional-full.csv'
df = pd.read_csv(DATA_PATH, sep=';')

print('Shape:', df.shape)
print()
print('Columns:', list(df.columns))
print()
print('Target distribution (count):')
print(df['y'].value_counts())
print()
print('Target distribution (%):')
print((df['y'].value_counts(normalize=True) * 100).round(2))

display(df.head())

In [ ]:
missing = df.isna().sum()
print('Missing values per column (top 10):')
print(missing.sort_values(ascending=False).head(10))

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols + ['y']]

print()
print('Numeric columns:', num_cols)
print()
print('Categorical columns:', cat_cols)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='y', ax=axes[0], palette='Set2')
axes[0].set_title('Target Distribution')

age_bins = pd.cut(df['age'], bins=AGE_BINS, include_lowest=True)
age_sub = df.assign(age_group=age_bins).groupby('age_group', observed=False)['y'].apply(lambda s: (s == 'yes').mean()).reset_index(name='subscribe_rate')
sns.barplot(data=age_sub, x='age_group', y='subscribe_rate', ax=axes[1], palette='Blues_d')
axes[1].set_title('Subscription Rate by Age Group')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Rate')

plt.tight_layout()
plt.show()


## Preprocessing and train/test split

In [ ]:

X = df.drop(columns=['y'])
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Train size:', X_train.shape, 'Test size:', X_test.shape)


## Utility functions

In [ ]:
results = {}
roc_data = {}
conf_matrices = {}


def evaluate_model(model_name, pipeline, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Train Time (s)': train_time,
    }

    results[model_name] = metrics
    conf_matrices[model_name] = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[model_name] = (fpr, tpr)

    print()
    print(f'{model_name} metrics:')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

    print()
    print('Classification report:')
    print(classification_report(y_test, y_pred, digits=4))

    return pipeline, metrics

## Baseline Logistic Regression

In [ ]:

baseline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='lbfgs'))
])

baseline_lr, baseline_metrics = evaluate_model('Baseline Logistic Regression', baseline_lr, X_train, y_train, X_test, y_test)


In [ ]:

plt.figure(figsize=(5,4))
sns.heatmap(conf_matrices['Baseline Logistic Regression'], annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Baseline Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## Hyperparameter Tuning

In [ ]:

param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs'],
    'model__max_iter': [500, 1000],
    'model__class_weight': [None, 'balanced']
}

tuned_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=RANDOM_STATE, penalty='l2'))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(
    tuned_lr,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=0,
    return_train_score=True
)

grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print('Best CV F1:', round(grid.best_score_, 4))


In [ ]:

cv_results = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score').reset_index(drop=True)
cv_results['step'] = np.arange(1, len(cv_results) + 1)

plt.figure(figsize=(10, 5))
plt.plot(cv_results['step'], cv_results['mean_test_score'], marker='o')
plt.title('Grid Search Optimization Steps (mean CV F1)')
plt.xlabel('Optimization step (sorted by rank)')
plt.ylabel('Mean CV F1')
plt.tight_layout()
plt.show()

display(cv_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))


In [ ]:

best_tuned_lr = grid.best_estimator_
best_tuned_lr, tuned_metrics = evaluate_model('Optimized Logistic Regression', best_tuned_lr, X_train, y_train, X_test, y_test)

q2_compare = pd.DataFrame([baseline_metrics, tuned_metrics], index=['Baseline', 'Optimized'])
q2_compare['F1 Improvement'] = q2_compare['F1'] - q2_compare.loc['Baseline', 'F1']
q2_compare['ROC-AUC Improvement'] = q2_compare['ROC-AUC'] - q2_compare.loc['Baseline', 'ROC-AUC']

display(q2_compare)


## Regularized Logistic Regression Models (L1 and L2)

In [ ]:

l1_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        random_state=RANDOM_STATE,
        penalty='l1',
        solver='liblinear',
        C=1.0,
        max_iter=1000
    ))
])

l2_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        random_state=RANDOM_STATE,
        penalty='l2',
        solver='lbfgs',
        C=1.0,
        max_iter=1000
    ))
])

l1_lr, l1_metrics = evaluate_model('L1 Logistic Regression', l1_lr, X_train, y_train, X_test, y_test)
l2_lr, l2_metrics = evaluate_model('L2 Logistic Regression', l2_lr, X_train, y_train, X_test, y_test)


## Logistic Model Comparison

In [ ]:

comparison_logistic = pd.DataFrame({
    'Baseline Logistic Regression': results['Baseline Logistic Regression'],
    'L1 Logistic Regression': results['L1 Logistic Regression'],
    'L2 Logistic Regression': results['L2 Logistic Regression']
}).T

display(comparison_logistic)

best_logistic_model_name = comparison_logistic['F1'].idxmax()
print('Best logistic model by F1:', best_logistic_model_name)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model_name in zip(axes, ['Baseline Logistic Regression', 'L1 Logistic Regression', 'L2 Logistic Regression']):
    sns.heatmap(conf_matrices[model_name], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'Confusion Matrix - {model_name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(8, 6))
for model_name in ['Baseline Logistic Regression', 'Optimized Logistic Regression', 'L1 Logistic Regression', 'L2 Logistic Regression']:
    fpr, tpr = roc_data[model_name]
    auc_value = results[model_name]['ROC-AUC']
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc_value:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()



**Interpretation:**
- The best model depends on the target metric. For imbalanced marketing data, F1 and recall are usually more informative than accuracy alone.
- L1 can improve sparsity and interpretability by shrinking some coefficients to zero.
- L2 usually provides more stable coefficients and can generalize better when many one-hot encoded features are present.


## Decision Threshold Optimization

In [ ]:

model_map = {
    'Baseline Logistic Regression': baseline_lr,
    'L1 Logistic Regression': l1_lr,
    'L2 Logistic Regression': l2_lr
}
selected_model_name = best_logistic_model_name
selected_model = model_map[selected_model_name]

required_thresholds = [0.2, 0.3, 0.4, 0.6, 0.7, 0.8]
thresholds = sorted(required_thresholds + [0.5])  # include 0.5 baseline reference
probs = selected_model.predict_proba(X_test)[:, 1]

threshold_results = []
for thr in thresholds:
    pred_thr = (probs >= thr).astype(int)
    threshold_results.append({
        'Threshold': thr,
        'Accuracy': accuracy_score(y_test, pred_thr),
        'Precision': precision_score(y_test, pred_thr, zero_division=0),
        'Recall': recall_score(y_test, pred_thr, zero_division=0),
        'F1': f1_score(y_test, pred_thr, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probs)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

best_threshold_row = threshold_df.loc[threshold_df['F1'].idxmax()]
print(f"Selected model: {selected_model_name}")
print(f"Best threshold by F1: {best_threshold_row['Threshold']}")


In [ ]:

plt.figure(figsize=(10, 5))
for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
    plt.plot(threshold_df['Threshold'], threshold_df[metric], marker='o', label=metric)

plt.title(f'Threshold Performance - {selected_model_name}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
plt.show()


## K-Nearest Neighbours Analysis and Comparison

In [ ]:

k_values = list(range(3, 51))
cv_f1_scores = []

for k in k_values:
    knn_pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(knn_pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_f1_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_f1_scores)]
print('Best K by CV F1:', best_k)


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(k_values, cv_f1_scores, marker='o')
plt.title('KNN Tuning: K vs Mean CV F1')
plt.xlabel('K')
plt.ylabel('Mean CV F1')
plt.tight_layout()
plt.show()


In [ ]:

knn_best = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier(n_neighbors=best_k))
])

knn_best, knn_metrics = evaluate_model('KNN (Best K)', knn_best, X_train, y_train, X_test, y_test)


In [ ]:

preprocessor_for_count = preprocessor.fit(X_train)
transformed_dim = preprocessor_for_count.transform(X_train.iloc[:1]).shape[1]

n_train = X_train.shape[0]
logistic_param_count = transformed_dim + 1  # coefficients + intercept
knn_stored_parameters = n_train * transformed_dim + n_train  # stored transformed vectors + labels (approx)

comparison_all = pd.DataFrame({
    'Baseline Logistic Regression': results['Baseline Logistic Regression'],
    'L1 Logistic Regression': results['L1 Logistic Regression'],
    'L2 Logistic Regression': results['L2 Logistic Regression'],
    'KNN (Best K)': results['KNN (Best K)']
}).T

comparison_all['Parameter Footprint'] = [
    logistic_param_count,
    logistic_param_count,
    logistic_param_count,
    knn_stored_parameters
]

display(comparison_all)
print('Transformed feature count:', transformed_dim)
print('Approx logistic trainable params:', logistic_param_count)
print('Approx KNN stored params:', knn_stored_parameters)



**KNN vs Logistic Regression (summary):**
- **Parameters**: Logistic regression stores a compact coefficient vector; KNN stores all training points.
- **Training time**: KNN training is usually fast (storage-based), but prediction can be slower on large data.
- **Scalability**: Logistic regression scales better for large datasets and real-time scoring.
- **Performance**: KNN can be competitive locally, while logistic regression is often more stable and interpretable.


## Client Segmentation and Feature Importance

In [ ]:

final_model_for_importance = model_map[selected_model_name]

feature_names = final_model_for_importance.named_steps['preprocessor'].get_feature_names_out()
coefficients = final_model_for_importance.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

display(coef_df.head(20))


In [ ]:

plt.figure(figsize=(10, 6))
top_coef = coef_df.head(20).sort_values('coefficient')
sns.barplot(data=top_coef, x='coefficient', y='feature', palette='vlag')
plt.title(f'Top 20 Coefficients ({selected_model_name})')
plt.tight_layout()
plt.show()


In [ ]:

seg_df = df.copy()
seg_df['subscribed'] = (seg_df['y'] == 'yes').astype(int)
seg_df['age_group'] = pd.cut(seg_df['age'], bins=AGE_BINS, include_lowest=True)

# Segment profile by age group + job with minimum support
segment_profile = (
    seg_df.groupby(['age_group', 'job'], observed=False)
    .agg(
        customers=('subscribed', 'size'),
        subscribe_rate=('subscribed', 'mean')
    )
    .reset_index()
)
segment_profile = segment_profile[segment_profile['customers'] >= 100]
segment_profile = segment_profile.sort_values('subscribe_rate', ascending=False)

display(segment_profile.head(15))


In [ ]:

plt.figure(figsize=(12, 6))
plot_seg = segment_profile.head(12).copy()
plot_seg['segment'] = plot_seg['age_group'].astype(str) + ' | ' + plot_seg['job'].astype(str)
sns.barplot(data=plot_seg, x='subscribe_rate', y='segment', palette='Greens_d')
plt.title('Top Client Segments by Subscription Rate (min 100 clients)')
plt.xlabel('Subscription Rate')
plt.ylabel('Segment')
plt.tight_layout()
plt.show()



### Business Insights and Recommendations

1. **Target high-propensity segments first** (top age/job groups by subscription rate) to improve campaign ROI.
2. **Use optimized calling thresholds** from Q5 depending on business objective:
   - Higher recall threshold policy (lower threshold) for maximizing conversions.
   - Higher precision threshold policy (higher threshold) for reducing unnecessary calls.
3. **Model choice**:
   - Prefer the best logistic model for scalability, stability, and interpretability.
   - Keep KNN as a benchmark or for local-pattern exploration.
4. **Operational use**:
   - Rank customers by predicted probability and prioritize outreach tiers.
   - Monitor drift over time and retrain periodically.


## Summary Tables

In [ ]:

print('Logistic model comparison:')
display(comparison_logistic)

print('All-model comparison (including KNN):')
display(comparison_all)

print('Threshold optimization table:')
display(threshold_df)
